# Reinforcement Learning: Frozen Lake

In the **CartPole** notebook you built a Q-learning agent from scratch. This notebook uses that *exact same* algorithm — the same update rule, the same explore-vs-exploit idea — on a new problem.

Two things change, and both make the ideas easier to see:

- **No more buckets.** CartPole handed you four decimals, so you had to chop them into buckets before the Q-table could hold them. Frozen Lake hands you a single whole number naming the square you stand on. The Q-table holds that directly.
- **You can look at the strategy.** A CartPole Q-table has thousands of rows and can't be drawn. Here the whole strategy fits on a 4x4 map, so by the end you will *see* the plan the agent worked out.

Along the way one genuinely new idea shows up: a **sparse reward**. In CartPole you scored `+1` every single step. Here you score `+1` only if you reach the goal, and `0` on every other step — so the agent has to work backwards from a rare success to figure out which earlier moves were good.

# The Frozen Lake

You are on a frozen lake, walking from the start `S` to the goal `G` to fetch a frisbee. Some squares are safe ice `F`; some are holes `H` that end the attempt.

```
S F F F
F H F H
F F F H
H F F G
```

- **Actions:** move `left`, `down`, `right`, `up` (0, 1, 2, 3).
- **Reward:** `+1` for stepping onto the goal. `0` for every other step — including falling in a hole.
- **The attempt ends** when you reach the goal, fall in a hole, or run out of steps.

The environment comes from **Gymnasium**, the same toolkit CartPole came from.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import gymnasium as gym
import numpy as np

# is_slippery=False for now: moving "right" really moves you right.
# We turn the slipperiness on later. No render_mode on this env - training will
# step it thousands of times, so it has to stay fast.
env = gym.make("FrozenLake-v1", is_slippery=False, disable_env_checker=True)

print("squares on the map :", env.observation_space.n)   # 16, numbered 0..15
print("possible actions   :", env.action_space.n)        # 4: left, down, right, up

# A Random Walker Has No Plan

Before any learning, here is the agent choosing moves completely at random. A second copy of the lake with `render_mode="human"` draws each step in the window below — Gymnasium paces the animation for you. Watch three attempts: the walker shuffles around and usually drops into a hole, with no reason to head for the goal.

In [ ]:
demo = gym.make("FrozenLake-v1", is_slippery=False, render_mode="human", disable_env_checker=True)

for attempt in range(3):
  state, _ = demo.reset()
  for step in range(20):
    state, reward, terminated, truncated, _ = demo.step(demo.action_space.sample())
    if terminated or truncated:
      break
  print(f"attempt {attempt + 1}: {step + 1} steps, reward {reward}")

demo.close()
print("no plan, no learning - a random walker almost never reaches the goal")

# What the Agent Sees

Every step the environment hands back a single integer: **which square you are on**, numbered `0` to `15` left-to-right, top-to-bottom.

```
 0  1  2  3
 4  5  6  7
 8  9 10 11
12 13 14 15
```

Square `6` is row `6 // 4 = 1`, column `6 % 4 = 2`. That's the whole observation — no decimals, nothing to bucket. Below is a helper that draws the map with the agent shown as `@` (`.` = ice, `#` = hole, `G` = goal).

In [ ]:
SYMBOL = {"S": ".", "F": ".", "H": "#", "G": "G"}
ACTION_NAME = {0: "left", 1: "down", 2: "right", 3: "up"}

def draw(state, env):
  """Print the lake with the agent shown as @ (. = ice, # = hole, G = goal)."""
  grid = env.unwrapped.desc.astype("U1")
  nrows, ncols = grid.shape
  pr, pc = divmod(state, ncols)
  for i in range(nrows):
    print(" ".join(
      "@" if (i, j) == (pr, pc) else SYMBOL[grid[i, j]]
      for j in range(ncols)
    ))
  print()

env.reset(seed=0)
draw(0, env)

# The Q-Table

The **Q-table** has one row per square and one column per action. Each entry answers:

> *If I take this action from this square, and then keep playing well, how much total reward do I expect?*

It starts as all zeros — the agent knows nothing. `16 squares x 4 actions = 64` numbers to fill in. (CartPole's table had many thousands.)

In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n
q_table = np.zeros((n_states, n_actions))

print("Q-table shape     :", q_table.shape)
print("numbers to fill in:", q_table.size)
print("all zero to start :", q_table.sum() == 0)

# The Learning Rule

This is the **same rule as CartPole**, unchanged:

```
Q[state, action] += lr * ( reward + gamma * max(Q[next_state]) - Q[state, action] )
```

- `reward` — what we just got. **Almost always `0` here.**
- `gamma * max(Q[next_state])` — how good the next square looks. `gamma` (0 to 1) sets how much the future counts.
- `lr` — the learning rate: how big a nudge each step gives.
- `epsilon` — chance of a random move instead of the best-known one. Starts near `1.0`, shrinks over training.

**Why the sparse reward matters.** For thousands of steps every `reward` is `0`, so every update just nudges toward `0` and nothing is learned. Only when a random walk *stumbles onto the goal* does a real `+1` enter the table. From there `gamma * max(Q[next_state])` carries that value backwards one square at a time, over many episodes, until squares far from the goal finally "know" which way to go. That slow backward spread is called **credit assignment**.

In [ ]:
LEARNING_RATE = 0.1
DISCOUNT = 0.99
MIN_EPSILON = 0.01
EPISODES = 5000

results = []
try:
  for episode in range(EPISODES):
    state, _ = env.reset()
    epsilon = max(MIN_EPSILON, 1.0 - episode / (EPISODES * 0.6))
    done = False
    while not done:
      if np.random.random() < epsilon:
        action = env.action_space.sample()         # explore
      else:
        action = int(np.argmax(q_table[state]))     # exploit

      next_state, reward, terminated, truncated, _ = env.step(action)
      done = terminated or truncated

      best_next = np.max(q_table[next_state]) * (not terminated)
      td_target = reward + DISCOUNT * best_next
      q_table[state, action] += LEARNING_RATE * (td_target - q_table[state, action])
      state = next_state

    results.append(reward)   # 1.0 if the goal was reached, else 0.0
    if (episode + 1) % 1000 == 0:
      recent = np.mean(results[-1000:])
      print(f"episode {episode + 1:5d}   epsilon {epsilon:.2f}   reached goal (last 1000): {recent:.0%}")
except KeyboardInterrupt:
  print(f"\nstopped early at episode {episode + 1}")

print("\ndone - on non-slippery ice a trained agent should reach the goal almost every time")

# Look at the Strategy

Now the payoff. Draw the map and, in every safe square, an arrow for the action with the highest Q-value — the agent's plan. The shading shows `max(Q[square])`: how good the agent thinks each square is. Bright near the goal, dark far away — you can see the `+1` having spread backwards.

In [ ]:
import matplotlib.pyplot as plt

ARROW_GLYPH = {0: "\u2190", 1: "\u2193", 2: "\u2192", 3: "\u2191"}  # left, down, right, up

def draw_policy(table, env, title):
  grid = env.unwrapped.desc.astype("U1")
  nrows, ncols = grid.shape
  values = table.max(axis=1).reshape(nrows, ncols)

  fig, ax = plt.subplots(figsize=(ncols + 1, nrows + 1))
  ax.imshow(values, cmap="Blues", vmin=0.0)
  for i in range(nrows):
    for j in range(ncols):
      s = i * ncols + j
      cell = grid[i, j]
      if cell == "H":
        text = "hole"
      elif cell == "G":
        text = "GOAL"
      else:
        text = ARROW_GLYPH[int(np.argmax(table[s]))]
      ax.text(j, i, text, ha="center", va="center", fontsize=18)
  ax.set_xticks([])
  ax.set_yticks([])
  ax.set_title(title)
  plt.show()

draw_policy(q_table, env, "Strategy on non-slippery ice")

# Watch It Cross the Lake

Run the trained agent with **no random moves** — always its best-known action — and watch it in the window below. Gymnasium draws the lake for us: safe ice, cracked ice for the holes, the goal, and the character on its current square. On solid ice the plan works every time.

In [ ]:
import gymnasium as gym   # importing gymnasium also wires up the in-browser display
import numpy as np

# render_mode="human" draws each step automatically, paced by Gymnasium.
show = gym.make("FrozenLake-v1", is_slippery=False, render_mode="human", disable_env_checker=True)
state, _ = show.reset(seed=0)

done = False
while not done:
  action = int(np.argmax(q_table[state]))
  state, reward, terminated, truncated, _ = show.step(action)
  done = terminated or truncated

show.close()
print("Reached the goal!" if reward == 1.0 else "Fell in a hole.")

# Now Make the Ice Slippery

Real ice is slippery. With `is_slippery=True`, your chosen action only happens **one third of the time** — the rest of the time you slide to one side instead. "Go right" might send you up or down.

Nothing about the agent changes. Same Q-table, same update rule, same code. Only the world got harder. Watch what that does to both the learning curve and the strategy.

Slippery ice needs many more attempts to learn from, so the next cell runs `20000` episodes and takes a little longer — give it a moment. Drag the `EPISODES` slider if you want to trade accuracy for speed.

In [ ]:
slippery = gym.make("FrozenLake-v1", is_slippery=True, disable_env_checker=True)

n_states = slippery.observation_space.n
n_actions = slippery.action_space.n
q_slip = np.zeros((n_states, n_actions))

LEARNING_RATE = 0.1
DISCOUNT = 0.99
MIN_EPSILON = 0.01
EPISODES = 20000  #@param {type:"slider", min:5000, max:50000, step:5000}

results = []
try:
  for episode in range(EPISODES):
    state, _ = slippery.reset()
    epsilon = max(MIN_EPSILON, 1.0 - episode / (EPISODES * 0.7))
    done = False
    while not done:
      if np.random.random() < epsilon:
        action = slippery.action_space.sample()
      else:
        action = int(np.argmax(q_slip[state]))

      next_state, reward, terminated, truncated, _ = slippery.step(action)
      done = terminated or truncated

      best_next = np.max(q_slip[next_state]) * (not terminated)
      td_target = reward + DISCOUNT * best_next
      q_slip[state, action] += LEARNING_RATE * (td_target - q_slip[state, action])
      state = next_state

    results.append(reward)
    if (episode + 1) % 2000 == 0:
      recent = np.mean(results[-2000:])
      print(f"episode {episode + 1:5d}   epsilon {epsilon:.2f}   reached goal (last 2000): {recent:.0%}")
except KeyboardInterrupt:
  print(f"\nstopped early at episode {episode + 1}")

In [ ]:
res = np.array(results)
window = 500
moving = np.convolve(res, np.ones(window) / window, mode="valid")

plt.figure(figsize=(9, 4))
plt.plot(range(window - 1, len(res)), moving, linewidth=2)
plt.xlabel("episode")
plt.ylabel(f"share of last {window} attempts that reached the goal")
plt.title("Learning on slippery ice")
plt.ylim(0, 1)
plt.show()

print(f"final success rate: {moving[-1]:.0%}")
print("it climbs, then flattens out well below 100% - the note below explains why")

In [ ]:
draw_policy(q_slip, slippery, "Strategy on slippery ice")

# Watch It on Slippery Ice

Same idea, slippery lake. The agent always picks its best-known move, but the ice has the final say. Run this cell a few times — some runs cruise to the goal, some slip into a hole with the exact same strategy.

In [ ]:
import gymnasium as gym
import numpy as np

show = gym.make("FrozenLake-v1", is_slippery=True, render_mode="human", disable_env_checker=True)
state, _ = show.reset()

for _ in range(50):
  action = int(np.argmax(q_slip[state]))
  state, reward, terminated, truncated, _ = show.step(action)
  if terminated or truncated:
    break

show.close()
print("Reached the goal!" if reward == 1.0 else "Slipped into a hole this time - run it again.")

One run is mostly luck. Run the *same* strategy 100 times and count how often it reaches the goal — that is the real measure of how good it is.

In [ ]:
runs = 100
wins = 0
for _ in range(runs):
  state, _ = slippery.reset()
  done = False
  while not done:
    action = int(np.argmax(q_slip[state]))
    state, reward, terminated, truncated, _ = slippery.step(action)
    done = terminated or truncated
  wins += int(reward == 1.0)

print(f"the trained agent reached the goal in {wins} of {runs} runs")
print("same strategy every run - the spread is the ice, not the agent")

# Why It Never Reaches 100%

On slippery ice, **even a perfect strategy only reaches the goal about 70-75% of the time.** One third of your moves go sideways, and next to a hole that is sometimes unrecoverable no matter how well you played. The agent isn't failing to learn — it has learned about as well as this lake allows.

Look back at the slippery strategy map and you'll often see arrows that seem wrong — pointing at a wall, or away from the goal. They're not wrong. Pushing *into* the wall beside a hole means that when you slip, you slide along the wall instead of into the hole. The agent discovered that hugging the edges is safer, purely from reward.

# Things to Try

- Drag `EPISODES` up to `50000` and retrain. Does the success rate climb much higher, or has it already plateaued?
- Set `MIN_EPSILON = 0.0` in the slippery training cell. Does never taking a random move near the end help or hurt?
- Change `DISCOUNT` to `0.9`. With the reward this far away, does a shorter-sighted agent still find the goal?
- Make a bigger lake: `gym.make("FrozenLake-v1", is_slippery=True, map_name="8x8", disable_env_checker=True)`. Re-run the training and policy cells. How many more episodes does it need?
- Design your own lake and pass it in:
  `gym.make("FrozenLake-v1", is_slippery=False, desc=["SFFF", "FFFH", "FHFF", "HFFG"], disable_env_checker=True)`

# Check Your Understanding